In [1]:
%pip install timm
%pip install ipywidgets
%pip install torchmetrics
%pip install pytorch_accelerated

  Using cached timm-0.9.2-py3-none-any.whl (2.2 MB)
  Using cached torch-2.0.1-cp311-none-macosx_11_0_arm64.whl (55.8 MB)
  Using cached torchvision-0.15.2-cp311-cp311-macosx_11_0_arm64.whl (1.4 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 1.5 MB/s eta 0:00:00a 0:00:01
  Using cached safetensors-0.3.1-cp311-cp311-macosx_12_0_arm64.whl (401 kB)
  Using cached filelock-3.12.2-py3-none-any.whl (10 kB)
  Using cached sympy-1.12-py3-none-any.whl (5.7 MB)
  Using cached networkx-3.1-py3-none-any.whl (2.1 MB)
  Using cached fsspec-2023.6.0-py3-none-any.whl (163 kB)
  Using cached tqdm-4.65.0-py3-none-any.whl (77 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

[notice] A new release of pip is available: 23.0.1 -> 23.2
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.0.2/libexec/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 1.9 MB/s 


[notice] A new release of pip is available: 23.0.1 -> 23.2
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.0.2/libexec/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.2/729.2 kB 2.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 23.2
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.0.2/libexec/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 3.7 MB/s eta 0:00:00a 0:00:01
  Using cached tqdm-4.64.1-py2.py3-none-any.whl (78 kB)
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.65.0
    Uninstalling tqdm-4.65.0:
      Successfully uninstalled tqdm-4.65.0

[notice] A new release of pip is available: 23.0.1 -> 

In [2]:
# from timm import list_models
import timm
import torch



In [3]:
len(timm.list_models(pretrained=True))

1163

### UI element classes 

We'll support the following elements in this training:

In [4]:
classifications = [
    "button",
    "image",
    "label",
#     "switchButton",
    "textField",
#     "textArea",
#     "activityIndicator",
    "navigationBar",
#     "line",
#     "listElement",
#     "list",
    "tabBar",
#     "segmentedControl",
#     "picker",
#     "map",
#     "pagingDots",
#     "containerBox",
#     "checkBox",
#     "radioButton"
]

num_classes = len(classifications)
print(num_classes)

6


In [5]:
model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=len(classifications), global_pool='avgmax')
model.eval()

MobileNetV3(
  (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): Hardswish()
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (bn1): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (se): Identity()
        (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): Identity()
        )
        (drop_path): Identity()
      )
    )
    (1): Sequential(
      (0): InvertedResidual(
 

In [6]:
model.get_classifier()
num_in_features = model.get_classifier().in_features
print(num_in_features)

1280


In [7]:
from torch import nn

model.classifier = nn.Sequential(
    nn.BatchNorm1d(num_in_features),
    nn.Linear(in_features=num_in_features, out_features=512, bias=False),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.4),
    nn.Linear(in_features=512, out_features=len(classifications), bias=False)
)

model.eval()

MobileNetV3(
  (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): Hardswish()
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (bn1): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (se): Identity()
        (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): Identity()
        )
        (drop_path): Identity()
      )
    )
    (1): Sequential(
      (0): InvertedResidual(
 

In [8]:
model(torch.randn(1, 3, 224, 224)).shape

torch.Size([1, 6])

#### Turn on Apple Silicon support

In [9]:
mps_device = torch.device("mps")
model.to(mps_device)

MobileNetV3(
  (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): Hardswish()
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (bn1): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (se): Identity()
        (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): Identity()
        )
        (drop_path): Identity()
      )
    )
    (1): Sequential(
      (0): InvertedResidual(
 

In [16]:
from timm.data.transforms_factory import create_transform
from timm.data import create_dataset

ds = create_dataset(name='', root='data', transform=create_transform(224, is_training=True), split='validation', batch_size = 2)
print(ds)

In [17]:
ds_iter = iter(ds)
image, label = next(ds_iter)
print(ds)

In [24]:
# print(ds[0])
print(len(ds))
split_dataset = torch.utils.data.random_split(ds, [0.9, 0.1])#[242, 44, 1])#[0.9, 0.1])
train, validation = [split_dataset[i] for i in (0, 1)]
print(len(train))
print(len(validation))
test_train = train
print(test_train)
# while len(test_train) % 2 == 1:
#     last = test_train[-1]
#     print(f"Dropping last: {last}")
#     test_train = test_train[:-1]
print(len(test_train))
# train_split, val_split = ds.train_test_split(test_size=0.1).values()
# len(train_split)

930
837
93
837


In [27]:
import timm.data
import timm.loss
import timm.optim
import timm.utils
import torch
import torchmetrics
from timm.scheduler import CosineLRScheduler

from pytorch_accelerated.callbacks import SaveBestModelCallback
from pytorch_accelerated.trainer import Trainer, DEFAULT_CALLBACKS

class TimmMixupTrainer(Trainer):
    def __init__(self, eval_loss_fn, mixup_args, num_classes, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.eval_loss_fn = eval_loss_fn
        self.num_updates = None
        self.mixup_fn = timm.data.Mixup(**mixup_args)

        self.accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)
        self.ema_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)
        self.ema_model = None

    def create_scheduler(self):
        return timm.scheduler.CosineLRScheduler(
            self.optimizer,
            t_initial=self.run_config.num_epochs,
            cycle_decay=0.5,
            lr_min=1e-6,
            t_in_epochs=True,
            warmup_t=3,
            warmup_lr_init=1e-4,
            cycle_limit=1,
        )

    def training_run_start(self):
        # Model EMA requires the model without a DDP wrapper and before sync batchnorm conversion
        self.ema_model = timm.utils.ModelEmaV2(
            self._accelerator.unwrap_model(self.model), decay=0.9
        )
        if self.run_config.is_distributed:
            self.model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(self.model)

    def train_epoch_start(self):
        super().train_epoch_start()
        self.num_updates = self.run_history.current_epoch * len(self._train_dataloader)

    def calculate_train_batch_loss(self, batch):
        xb, yb = batch
        mixup_xb, mixup_yb = self.mixup_fn(xb, yb)
        return super().calculate_train_batch_loss((mixup_xb, mixup_yb))

    def train_epoch_end(
        self,
    ):
        self.ema_model.update(self.model)
        self.ema_model.eval()

        if hasattr(self.optimizer, "sync_lookahead"):
            self.optimizer.sync_lookahead()

    def scheduler_step(self):
        self.num_updates += 1
        if self.scheduler is not None:
            self.scheduler.step_update(num_updates=self.num_updates)

    def calculate_eval_batch_loss(self, batch):
        with torch.no_grad():
            xb, yb = batch
            outputs = self.model(xb)
            val_loss = self.eval_loss_fn(outputs, yb)
            self.accuracy.update(outputs.argmax(-1), yb)

            ema_model_preds = self.ema_model.module(xb).argmax(-1)
            self.ema_accuracy.update(ema_model_preds, yb)

        return {"loss": val_loss, "model_outputs": outputs, "batch_size": xb.size(0)}

    def eval_epoch_end(self):
        super().eval_epoch_end()

        if self.scheduler is not None:
            self.scheduler.step(self.run_history.current_epoch + 1)

        self.run_history.update_metric("accuracy", self.accuracy.compute().cpu())
        self.run_history.update_metric(
            "ema_model_accuracy", self.ema_accuracy.compute().cpu()
        )
        self.accuracy.reset()
        self.ema_accuracy.reset()

# Set training arguments, hardcoded here for clarity
image_size = (224, 224)
lr = 5e-3
smoothing = 0.1
mixup = 0.2
cutmix = 1.0
batch_size = 2#32
bce_target_thresh = 0.2
num_epochs = 40

mixup_args = dict(
    mixup_alpha=mixup,
    cutmix_alpha=cutmix,
    label_smoothing=smoothing,
    num_classes=num_classes,
)

# Create optimizer
optimizer = timm.optim.create_optimizer_v2(
    model, opt="adam", lr=lr, weight_decay=0.01
)

# As we are using Mixup, we can use BCE during training and CE for evaluation
train_loss_fn = timm.loss.BinaryCrossEntropy(
    target_threshold=bce_target_thresh, smoothing=smoothing
)
validate_loss_fn = torch.nn.CrossEntropyLoss()

print(optimizer)

# Create trainer and start training
trainer = TimmMixupTrainer(
    model=model,
    optimizer=optimizer,
    loss_func=train_loss_fn,
    eval_loss_fn=validate_loss_fn,
    mixup_args=mixup_args,
    num_classes=num_classes,
    callbacks=[
        *DEFAULT_CALLBACKS,
        SaveBestModelCallback(watch_metric="accuracy", greater_is_better=True),
    ],
)

trainer.train(
    per_device_batch_size=batch_size,
    train_dataset=train,
    eval_dataset=validation,
    num_epochs=num_epochs,
    create_scheduler_fn=trainer.create_scheduler,
)

PYTORCH_ENABLE_MPS_FALLBACK=1
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0.0

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0.01
)

Starting training run

Starting epoch 1



  0%|                                                   | 0/419 [00:00<?, ?it/s]

NotImplementedError: The operator 'aten::hardsigmoid.out' is not currently implemented for the MPS device. If you want this op to be added in priority during the prototype phase of this feature, please comment on https://github.com/pytorch/pytorch/issues/77764. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.

In [13]:
EPOCH = 40
PATH = "model.pt"
LOSS = 2.48

torch.save({
            'epoch': EPOCH,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': LOSS,
            }, PATH)

In [27]:
model.eval()
data = torch.randn(1, 3, 24, 24) # Load your data here, this is just dummy data
output = model(data)
prediction = torch.argmax(output)
print(prediction)

tensor(1)


### Attempt a test image

In [58]:
from PIL import Image
from torchvision import transforms

from torchvision.io import read_image, ImageReadMode

test_image = read_image("random_button.png", mode=ImageReadMode.RGB)

file = Image.open("random_button.png")#.convert('RGB')
# print(file.shape)
transforms = transforms.ToTensor()
input = transforms(file)
print(input.shape)
transform2 = create_transform(224, is_training=False)
input2 = transform2(file.convert('RGB'))
print(input2.shape)

output = model(input2.unsqueeze(0))
print(output)
prediction = torch.argmax(output)
print(prediction)

torch.Size([4, 100, 220])
torch.Size([3, 224, 224])
tensor([[-4.1487, -0.3889, -2.8478, -4.1214, -0.5247, -2.2242, -3.8456, -4.4121,
         -4.0928, -3.4406, -4.0615, -4.0001, -4.2164, -3.6672, -4.4692, -3.4874,
         -4.2008, -4.2130, -4.2257]], grad_fn=<MmBackward0>)
tensor(1)


In [14]:
%pip install coremltools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 941.9/941.9 kB 9.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 13.4 MB/s eta 0:00:00
  Created wheel for coremltools: filename=coremltools-6.3.0-py3-none-any.whl size=1220313 sha256=3e1e6f2b84ff62eda93a23b15a66ad31b8ddfa78eaed40f8e90377e8e3fd9637
  Stored in directory: <pip-cache>Successfully built coremltools
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.21.12
    Not uninstalling protobuf at /opt/homebrew/lib/python3.11/site-packages, outside environment /opt/homebrew/Cellar/jupyterlab/3.6.3/libexec
    Can't uninstall 'protobuf'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.54.2 requires protobuf>=4.21.6, but you have protobuf 3.20.3 which is incompa

In [16]:
# Trace the model with random data.
cpu_device = torch.device("cpu")
model.to(cpu_device)

example_input = torch.rand(1, 3, 224, 224) 
traced_model = torch.jit.trace(model, example_input)
out = traced_model(example_input)

In [18]:
import coremltools as ct

# Using image_input in the inputs parameter:
# Convert to Core ML program using the Unified Conversion API.
model = ct.convert(
    traced_model,
    convert_to="mlprogram",
    inputs=[ct.TensorType(shape=example_input.shape)]
)

# Save the converted model.
model.save("newmodel.mlpackage")

Converting PyTorch Frontend ==> MIL Ops: 100%|▉| 498/499 [00:00<00:00, 8525.67 o
Running MIL frontend_pytorch pipeline: 100%|█| 5/5 [00:00<00:00, 478.19 passes/s
Running MIL default pipeline: 100%|████████| 57/57 [00:01<00:00, 55.27 passes/s]
Running MIL backend_mlprogram pipeline: 100%|█| 10/10 [00:00<00:00, 984.93 passe


RuntimeError: BlobWriter not loaded